In [1]:
# %%capture
# !pip install snntorch tqdm matplotlib tensorboard qiskit optuna
# !pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# import sys
# !pip install dill==0.3.9 --target "/mnt/user/shared/NMC test collab for user jsvanleeuwen/BrainScaleS/QSVAE/localpackages" 
# !pip3 install qiskit_aer 
# sys.path.append('local_python_packages')

In [2]:
# sys.path.insert(0,'/mnt/user/shared/NMC test collab for user jsvanleeuwen/BrainScaleS/QSVAE/localpackages')

In [3]:
# !pip3 uninstall -y dill

In [4]:
# import sys
# print(sys.executable)

In [5]:

# import pkg_resources
# pkg_resources.require("dill==0.3.9")
# import dill
# print(dill.__version__)
# dill.__file__

In [6]:
# import sys
# print(sys.version)
# print(sys.path)

In [7]:
# import os, sys
# os.path.dirname(sys.executable)

In [8]:
# from typing import Tuple
# import matplotlib.pyplot as plt
# import ipywidgets as w
# import numpy as np
# import torch
from src._static.common.helpers import setup_hardware_client, save_nightly_calibration
# from src._static.tutorial.snn_yinyang_helpers import plot_data, plot_input_encoding, plot_training
while(True):
    try:
        setup_hardware_client()
    except Exception as e:
        print("Error: ", e)
    else:
        break

INFO  10:11:40,621  demo_helpers Connection to hxcube7fpga0chip57_1 established


In [9]:
%load_ext autoreload

In [10]:
%autoreload 2

In [11]:
# hxtorch.release_hardware()
# from src.scaling import get_trace_scaling
# from hxtorch.spiking.utils import calib_helper

# print(get_trace_scaling(1,1,"spiking2_cocolist.pbin"))

In [12]:
import torch
import hxtorch
%matplotlib inline
from src.Quantum_circuits import *
import numpy as np

from src.QSVAE_model import *
from src.POVM_dataset import *
from src.SNN_brainscales import *
import gc
import time
import json
import traceback
from datetime import datetime
import optuna

def save_progress(epoch, iteration, filename="progress.json"):
    with open(filename, "w") as f:
        json.dump({"epoch": epoch, "iteration": iteration}, f)

def load_progress(filename="progress.json"):
    try:
        with open(filename, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        return {"epoch": 0, "iteration": 0}
# save_progress(0, 0)

log = hxtorch.logger.get("grenade.backend")
hxtorch.logger.default_config(level=hxtorch.logger.LogLevel.ERROR)

if torch.cuda.is_available():
 dev = "cuda:0"
else:
 dev = "cpu"
device = torch.device(dev)
print(device)
!python --version
torch.set_num_threads(8)  # Adjust based on your CPU cores


class QSVAE_Params:
    def __init__(self, 
                 I, sigma_x, sigma_y, sigma_z, s_vectors,
                 n, shots, first_run, backend_type, train, test, val,
                 beta, num_steps, num_epochs, learning_rate,
                 batch_train, batch_test, batch_val, num_workers,
                 shuffle, split, device, input_size, hidden_size, 
                 output_size, mock, alpha, result=None, circuits=None, backend=None, probabilities=None):
        self.I = I
        self.sigma_x = sigma_x
        self.sigma_y = sigma_y
        self.sigma_z = sigma_z
        self.s_vectors = s_vectors
        self.n = n
        self.shots = shots
        self.first_run = first_run
        self.backend_type = backend_type
        self.train = train
        self.test = test
        self.val = val
        self.beta = beta
        self.num_steps = num_steps
        self.num_epochs = num_epochs
        self.learning_rate = learning_rate
        self.batch_train = batch_train
        self.batch_test = batch_test
        self.batch_val = batch_val
        self.num_workers = num_workers
        self.shuffle = shuffle
        self.split = split
        self.device = device
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.hidden_size_cpu = hidden_size_cpu
        self.mock = mock
        self.alpha = alpha
        self.load_model = load_model
        self.result = result
        self.circuits = circuits
        self.backend = backend
        self.probabilities = probabilities
    

# Define the Pauli matrices
I = torch.eye(2, dtype=torch.complex64)
sigma_x = torch.tensor([[0, 1], [1, 0]], dtype=torch.complex64)
sigma_y = torch.tensor([[0, -1j], [1j, 0]], dtype=torch.complex64)
sigma_z = torch.tensor([[1, 0], [0, -1]], dtype=torch.complex64)

# Define the s^(alpha) vectors for the single-qubit POVM
s_vectors = [
    torch.tensor([0, 0, 1], dtype=torch.float32),
    torch.tensor([2 * np.sqrt(2) / 3, 0, -1 / 3], dtype=torch.float32),
    torch.tensor([-np.sqrt(2) / 3, np.sqrt(2) / 3, -1 / 3], dtype=torch.float32),
    torch.tensor([-np.sqrt(2) / 3, -np.sqrt(2) / 3, -1 / 3], dtype=torch.float32)
]

# Parameters
n = 3
shots = 100_000
first_run = False
load_model = False
backend_type = "AerSimulator"
train = True
test = False
val = True
beta = 0.819
num_steps = 100
num_epochs = 5
learning_rate = 1e-2
batch_train, batch_test, batch_val = (300, 200, 1_000_0)
num_workers = 0
shuffle = False
split = [0.8, 0.2, 0]
input_size = 4 * n
hidden_size = 20 * n
output_size = 2 * 2**n
hidden_size_cpu = 20 * n 
mock = False
alpha = 1
model_recover = False

# result, circuits, backend = None, None, select_backend(backend_type)
# quantum_exp = QuantumExperiment(backend, n, shots)
# result, circuits = quantum_exp.run_experiment()

# Create an instance of QSVAE_Params
params = QSVAE_Params(
    I, sigma_x, sigma_y, sigma_z, s_vectors,
    n, shots, first_run, backend_type, train, test,
    val, beta, num_steps, num_epochs, learning_rate, batch_train,
    batch_test, batch_val, num_workers, shuffle, split, device,
    input_size, hidden_size, output_size, hidden_size_cpu, mock, alpha, load_model
)

# /srv/main-spack-instance-2402/spack/var/spack/environments/experimental/.spack-env/view/lib/

/srv/main-spack-instance-2402/spack/var/spack/environments/experimental/.spack-env/view/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: ''If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


cpu
Python 3.8.11


In [14]:
parameters = [
    (3, 300, 20_000),
    (4, 300, 100_000),
    # (5, 600, 4**5 * 500),  # 4^5 * 500
    # (6, 600, 4**6 * 500),  # 4^6 * 500
    # (7, 600, 4**7 * 500),  # 4^7 * 500
    # (8, 1000, 4**8 * 500)  # 4^8 * 500
]
# parameters = [
#     (3, 5000, 20_000),
#     (4, 5000, 100_000),
#     (5, 5000, 4**5 * 500),  # 4^5 * 500
#     (6, 5000, 4**6 * 500),  # 4^6 * 500
#     (7, 5000, 4**7 * 500),  # 4^7 * 500
#     (8, 5000, 4**8 * 500)  # 4^8 * 500
# ]

date = datetime.now().strftime("%Y-%m-%d")

def initialize_hardware(max_retries=5, wait_time=5):
    retries = 0
    while retries < max_retries:
        gc.collect()
        try:
            print("hxtorch loading")
            hxtorch.init_hardware() 
            print("hxtorch loaded successfully")
            return True
        except Exception as e:
            print(f"hxtorch error: {e}. Retrying ({retries+1}/{max_retries})...")
            hxtorch.release_hardware()
            retries += 1
            time.sleep(wait_time)
    print("Failed to initialize hardware after maximum retries.")
    return False

while(True):
    try:
        if initialize_hardware():

            # Run the model for each parameter setting and calculate fidelity:
            fidelities = []
            for param in parameters:
                params.n, params.batch_train, params.batch_val = param
                params.input_size = 4 * params.n
                params.hidden_size = 20 * params.n
                params.output_size = 2 * 2**params.n
                # Load data
                POVM_dataset = load_data(params)

                # Instantiate the model
                model = SQVAE(params, POVM_dataset)
                
                # Load params if model needs to be recovered
                try:
                    model.model.load_state_dict(torch.load(f"data/model_recover_point_{date}_{params.n}.pt", weights_only=True))
                except Exception as e:
                    print("New model training started", e)
                    save_progress(0, 0)
                finally:
                    fidelity_score = model.run(params)
                
                            # Append the fidelity score to the list
                fidelities.append(fidelity_score)
                torch.cuda.empty_cache() 

            # Plot the histogram of fidelities
            plot_histogram(fidelities, parameters)
            
        else:
            print("Hardware initialization failed. Exiting.")
    except Exception as e:
        print("An exception occurred:", e)
        print(traceback.format_exc())
        torch.save(model.model.state_dict(), f"data/model_recover_point_{date}_{n}.pt")
        model_recover = True
        print("Saved model")
    else:
        print("Training done!")
        break
    finally:
        # Release the hardware connection
        hxtorch.release_hardware()
        gc.collect()
    
    

hxtorch loading
hxtorch loaded successfully
Dataset loaded.
New model training started [Errno 2] No such file or directory: 'data/model_recover_point_2025-01-08_3.pt'
Start epoch:  0
start_iteration:   0


Iterations:   4%|▎         | 10/267 [00:47<20:22,  4.76s/it]

 mse_loss is 2.628999948501587
 mmd_loss is 0.1289999932050705
 total loss = 2.628999948501587
The fidelity for 3 qubits is 0.25 with 300 samples.
Spike count encoder = 0.0 = 0.0%, spike count latent z = 0.0 = 0.0% 


Iterations:   7%|▋         | 20/267 [01:31<17:50,  4.33s/it]

 mse_loss is 2.615999937057495
 mmd_loss is 0.11599999666213989
 total loss = 2.615999937057495
The fidelity for 3 qubits is 0.25 with 300 samples.
Spike count encoder = 0.0 = 0.0%, spike count latent z = 0.0 = 0.0% 


...

 mse_loss is 2.500999927520752
 mmd_loss is 0.0010000000474974513
 total loss = 2.500999927520752
The fidelity for 3 qubits is 0.25 with 300 samples.
Spike count encoder = 0.0 = 0.0%, spike count latent z = 0.0 = 0.0% 


Iterations:  94%|█████████▎| 250/267 [17:29<01:12,  4.25s/it]

 mse_loss is 2.500999927520752
 mmd_loss is 0.0010000000474974513
 total loss = 2.500999927520752
The fidelity for 3 qubits is 0.25 with 300 samples.
Spike count encoder = 0.0 = 0.0%, spike count latent z = 0.0 = 0.0% 


Iterations:  97%|█████████▋| 258/267 [18:05<00:37,  4.21s/it]


KeyboardInterrupt: 

In [ ]:
r = torch.tensor([0.9,0.1])
print(r[1])
spk = torch.Tensor(4,3,2)    
u = torch.rand(spk.shape)
print(u[1,:,1])
z = (u < r.unsqueeze(0)).float()
print(z[1,:,1])

In [ ]:
# from src.QSVAE_model import *
# from src.POVM_dataset import *
# from src.SNN_brainscales import *
# import gc
# gc.collect()

# try:
#     hxtorch.init_hardware() 
# except:
#     print("hxtorch error")
#     hxtorch.release_hardware()
# else:
# for n in range(2, 9):
#     params.n = n
#     quantum_exp = QuantumExperiment(backend, params.n, params.shots)
#     # params.result, params.circuits = quantum_exp.run_experiment()
#     params.probabilities = quantum_exp.probabilities

#     POVM_dataset = load_data(params)

#     model = SQVAE(params, POVM_dataset)